# Práctica del martes resuelta y explicada

Copia didáctica, revisada el 30 de septiembre de 2026. Ejecuta en orden. Completa diagnóstico, detector, presupuesto de pasos/costo/repetición, verificador y Mini-Reflexion con feedback efectivamente recibido. Costos ficticios y búsquedas simuladas; biblioteca estándar de Python.


# S3·MAR — Trazas ReAct: leerlas, protegerlas, mejorarlas

**MMIA 6013 · Semana 3 · Martes**

1. Ejecutar un ReAct simulado y leer sus trazas
2. Implementar un detector de loops
3. Implementar presupuesto de pasos
4. Mini-Reflexion: reintentar con la crítica del verificador

Todo local y determinista — el foco es la *mecánica* de los patrones.

## 1. Qué programa estás construyendo

El lunes completaste el bucle que ejecuta herramientas. El martes examinas sus fallas y construyes tres controles: un detector de repetición, un presupuesto externo y un procedimiento que verifica y corrige salidas.

`json` convierte texto JSON en datos. `re` reconoce palabras para comparar solicitudes. Las trazas son listas ya escritas: imprimirlas no realiza búsquedas ni llama a un modelo. Los eventos `thought` son textos públicos del ejemplo, no acceso al razonamiento privado de un LLM.

Define funciones primero, ejecútalas después y compara la salida con lo previsto. Todas las celdas de esta copia usan biblioteca estándar y datos ficticios.

### Cómo recorrer la práctica sin perderte

Primero lee la explicación de una celda; después mira el código y finalmente su salida guardada. Si ejecutas el notebook, empieza por `import json` e `import re` y continúa de arriba abajo. Ejecutar una función antes de la celda que la define provoca `NameError`. Para repetir la verificación completa, reinicia el kernel y ejecuta todas las celdas: así no dependes de variables de una corrida anterior.

Una línea `def detectar_loop(...):` **define** qué hará una función; todavía no procesa ninguna traza. La línea `detectar_loop(TRAZA_LOOP)` **la llama** y entrega una traza concreta. `return` entrega un resultado y termina esa llamada; `print` lo muestra en pantalla. No son intercambiables.

| Función | Qué recibe | Qué devuelve | Para qué sirve |
| --- | --- | --- | --- |
| `imprimir_traza` | Lista de eventos. | `None`; escribe texto en la salida. | Leer lo ocurrido. |
| `normalizar_input` | Texto de una consulta. | Tupla de palabras ordenadas. | Comparar entradas. |
| `detectar_loop` | Traza y tamaño de ventana. | `True` o `False`. | Señalar repetición reciente. |
| `controlar_agente` | Acciones y límites. | Diccionario con estado, pasos, gasto y traza. | Bloquear acciones antes de exceder límites. |
| `verificador` | Texto candidato. | Pareja `(ok, critica)`. | Comprobar el contrato JSON. |
| `reflexion` | Número de intentos y función generadora opcional. | Diccionario con estado, respuesta y traza. | Reintentar con feedback verificable. |


In [1]:
import json
import re

## Parte 1 — Tres trazas ReAct para diagnosticar

Cada traza es una lista de eventos Thought / Action / Observation / Answer.

## 2. Una traza es una secuencia de eventos

Cada evento es un diccionario cuyo campo `tipo` dice cómo interpretarlo:

En Python, `[...]` crea una lista y `{...}` con pares `clave: valor` crea un diccionario. `TRAZA_LOOP[1]` accede al segundo evento, porque el primer índice es 0. `TRAZA_LOOP[1]["input"]` accede después al texto de entrada de ese diccionario.

El `for e in traza` visita un evento por vuelta. La condición `e["tipo"] == "action"` elige cómo imprimirlo. Dos signos `==` comparan valores; un solo `=` asigna un valor a una variable. Cada evento del ejemplo posee los campos que usa su rama: una acción tiene `tool` e `input`; una observación tiene `texto`.

| Tipo | Qué representa | Campos importantes |
| --- | --- | --- |
| `thought` | Una explicación escrita para el ejemplo. | `texto`. |
| `action` | Una petición de herramienta. | `tool` e `input`. |
| `observation` | El resultado recibido de una herramienta. | `texto`. |
| `answer` | Una respuesta final. | `texto`. |

`imprimir_traza` recorre la lista. Para una acción imprime el nombre de herramienta y su entrada; para los demás eventos imprime el texto. `!r` muestra la representación de una cadena con comillas; `capitalize()` pone su primera letra en mayúscula; `:12s` reserva un ancho mínimo para alinear la salida.

La traza sana del material realiza dos búsquedas diferentes antes de responder. La resta de años es solo una aproximación; para una edad exacta haría falta comparar fechas completas. El ejercicio estudia cómo la respuesta se relaciona con lo observado, no ejecuta una investigación biográfica.


In [2]:
TRAZA_SANA = [
    {"tipo": "thought", "texto": "Necesito saber quién fundó SpaceX y en qué año."},
    {"tipo": "action", "tool": "buscar", "input": "fundación SpaceX año fundador"},
    {"tipo": "observation", "texto": "SpaceX fue fundada en 2002 por Elon Musk."},
    {"tipo": "thought", "texto": "Ahora necesito el año de nacimiento de Musk."},
    {"tipo": "action", "tool": "buscar", "input": "Elon Musk año de nacimiento"},
    {"tipo": "observation", "texto": "Elon Musk nació el 28 de junio de 1971."},
    {"tipo": "thought", "texto": "2002 - 1971 = 31 (o 30 antes de junio). Puedo responder."},
    {"tipo": "answer", "texto": "Tenía alrededor de 30-31 años."},
]

TRAZA_LOOP = [
    {"tipo": "thought", "texto": "Debo encontrar el precio de la acción de NimbusSoft."},
    {"tipo": "action", "tool": "buscar", "input": "precio acción NimbusSoft"},
    {"tipo": "observation", "texto": "Sin resultados."},
    {"tipo": "thought", "texto": "Intentaré con otras palabras."},
    {"tipo": "action", "tool": "buscar", "input": "NimbusSoft precio acción"},
    {"tipo": "observation", "texto": "Sin resultados."},
    {"tipo": "thought", "texto": "Probaré de nuevo."},
    {"tipo": "action", "tool": "buscar", "input": "precio acción NimbusSoft"},
    {"tipo": "observation", "texto": "Sin resultados."},
    {"tipo": "thought", "texto": "Intentaré con otras palabras."},
    {"tipo": "action", "tool": "buscar", "input": "NimbusSoft precio acción"},
    {"tipo": "observation", "texto": "Sin resultados."},
]

TRAZA_FABRICA = [
    {"tipo": "thought", "texto": "Necesito los ingresos de NimbusSoft en 2025."},
    {"tipo": "action", "tool": "buscar", "input": "ingresos NimbusSoft 2025"},
    {"tipo": "observation", "texto": "NimbusSoft es una empresa de software fundada en 2019."},
    {"tipo": "thought", "texto": "Según los resultados, los ingresos fueron 45 millones."},  # ← ¡nadie dijo eso!
    {"tipo": "answer", "texto": "Los ingresos de NimbusSoft en 2025 fueron de 45 millones USD."},
]

def imprimir_traza(traza):
    for e in traza:
        if e["tipo"] == "action":
            print(f"  Action:      {e['tool']}({e['input']!r})")
        else:
            print(f"  {e['tipo'].capitalize():12s} {e['texto']}")

print("=== TRAZA SANA ==="); imprimir_traza(TRAZA_SANA)

=== TRAZA SANA ===
  Thought      Necesito saber quién fundó SpaceX y en qué año.
  Action:      buscar('fundación SpaceX año fundador')
  Observation  SpaceX fue fundada en 2002 por Elon Musk.
  Thought      Ahora necesito el año de nacimiento de Musk.
  Action:      buscar('Elon Musk año de nacimiento')
  Observation  Elon Musk nació el 28 de junio de 1971.
  Thought      2002 - 1971 = 31 (o 30 antes de junio). Puedo responder.
  Answer       Tenía alrededor de 30-31 años.


### ✏️ Ejercicio 1.1 (diagnóstico manual)
Imprime las otras dos trazas y anota en el comentario: ¿dónde exactamente se
rompe cada una? ¿Qué protección la habría atrapado?

## 3. Diagnóstico manual: repetición y fabricación no son el mismo fallo

En `TRAZA_LOOP`, la primera acción pide `precio acción NimbusSoft`. Después de «sin resultados», la segunda pide las mismas palabras en otro orden. Se repite una estrategia sin información nueva; vuelve a hacerlo dos veces más.

En `TRAZA_FABRICA`, la observación solo dice que la empresa es de software y fue fundada en 2019. El evento siguiente introduce «45 millones» y la respuesta final repite la cifra. **La cifra no procede de la observación**.

| Traza | Dónde aparece el problema | Protección pertinente |
| --- | --- | --- |
| Loop | La segunda acción repite la búsqueda normalizada; índices 1 y 4, contando desde 0. | Detector de repetición y límite de pasos/costo. |
| Fabricación | El `thought` de índice 3 introduce una cifra sin respaldo. | Comprobar evidencia y abstenerse cuando falta. |

Un detector de acciones repetidas no comprueba la verdad de los resultados. Por eso puede detectar el loop y no la fabricación.


In [3]:
imprimir_traza(TRAZA_LOOP)
imprimir_traza(TRAZA_FABRICA)
# LOOP: en el segundo action ya se repite la misma búsqueda con palabras reordenadas.
# FABRICA: el thought que añade "45 millones" no está respaldado por la observation.
# Detector + límites contienen repetición; verificación de evidencia trata fabricación.


  Thought      Debo encontrar el precio de la acción de NimbusSoft.
  Action:      buscar('precio acción NimbusSoft')
  Observation  Sin resultados.
  Thought      Intentaré con otras palabras.
  Action:      buscar('NimbusSoft precio acción')
  Observation  Sin resultados.
  Thought      Probaré de nuevo.
  Action:      buscar('precio acción NimbusSoft')
  Observation  Sin resultados.
  Thought      Intentaré con otras palabras.
  Action:      buscar('NimbusSoft precio acción')
  Observation  Sin resultados.
  Thought      Necesito los ingresos de NimbusSoft en 2025.
  Action:      buscar('ingresos NimbusSoft 2025')
  Observation  NimbusSoft es una empresa de software fundada en 2019.
  Thought      Según los resultados, los ingresos fueron 45 millones.
  Answer       Los ingresos de NimbusSoft en 2025 fueron de 45 millones USD.


## Parte 2 — Detector de loops

### ✏️ Ejercicio 2.1
Implementa `detectar_loop(traza, ventana=6)` que devuelva True si hay
acciones repetidas (misma tool + input equivalente) dentro de la ventana.
Normaliza el input: minúsculas y orden alfabético de palabras — así
"precio acción NimbusSoft" ≡ "NimbusSoft precio acción".

## 4. Detector: comparar la herramienta y las palabras de su entrada

`normalizar_input` transforma una cadena en una representación comparable. Para `precio acción NimbusSoft`:

```text
lower() -> precio acción nimbussoft
findall -> ['precio', 'acción', 'nimbussoft']
sorted -> ['acción', 'nimbussoft', 'precio']
tuple  -> ('acción', 'nimbussoft', 'precio')
```

`re.findall(r"\w+", ...)` encuentra grupos de caracteres de palabra. La tupla permite usar el resultado como parte de una clave en un conjunto. Reordenar las palabras produce la misma clave normalizada.

`traza[-ventana:]` selecciona los últimos eventos; con `ventana=6`, son **seis eventos**, no seis acciones. El detector ignora los que no son `action` mediante `continue`.

Para cada acción construye `(tool, input_normalizado)`. Si esa pareja ya está en `vistos`, devuelve `True`. Si no, la agrega. Solo devuelve `False` después de revisar toda la ventana.

| Entrada | Resultado esperado |
| --- | --- |
| `TRAZA_SANA` | `False`: las dos búsquedas tienen entradas distintas. |
| `TRAZA_LOOP` | `True`: la ventana contiene acciones equivalentes. |
| `TRAZA_FABRICA` | `False`: hay una sola búsqueda. |

La herramienta también forma parte de la clave: dos herramientas diferentes con el mismo texto no se consideran la misma acción.

### Seguir el detector con dos acciones

`vistos = set()` crea un conjunto vacío: conserva claves únicas y permite preguntar `clave in vistos`. No se escribe `{}` porque eso crearía un diccionario vacío.

1. Al encontrar la primera búsqueda, su clave es `("buscar", ("acción", "nimbussoft", "precio"))`. El conjunto está vacío: se añade con `vistos.add(clave)`.
2. Una observación no es una acción: `continue` salta a la siguiente vuelta sin crear una clave.
3. La segunda búsqueda cambia el orden de palabras, pero produce exactamente la misma clave. `clave in vistos` es verdadero y la función termina con `True`.

La ventana delimita **qué parte de la lista se revisa en esta llamada**. En `TRAZA_LOOP`, los últimos seis eventos contienen las acciones de índices 7 y 10. Con `ventana=3`, solo queda una acción y el resultado es `False`, aunque hubo repetición antes. Ampliar a seis vuelve a devolver `True`.

`detector_resultados` usa una comprensión de diccionario: por cada par `nombre, traza` de `.items()` ejecuta el detector y guarda `nombre: resultado`. Equivale a crear un diccionario vacío y llenarlo en un `for`.

El patrón `r"\w+"` es una cadena para una expresión regular: `r` conserva la barra inversa y `+` pide uno o más caracteres de palabra. Reconoce letras, dígitos y guiones bajos; omite la puntuación. Conserva tildes: `acción` y `accion` siguen siendo palabras distintas. No calcula significado.

### Límites de esta regla

«Cotización de NimbusSoft» no tiene las mismas palabras que «precio acción NimbusSoft», aunque pueda intentar lo mismo. En cambio, «Ana paga a Luis» y «Luis paga a Ana» tienen las mismas palabras, pero representan relaciones diferentes. Ordenar pierde esa diferencia.

Repetición tampoco demuestra siempre un error: otra consulta puede ser legítima si cambió el dato externo. Este detector implementa la regla del ejercicio, no una prueba universal de ausencia de progreso.


In [4]:
def normalizar_input(texto):
    palabras = re.findall(r"\w+", texto.lower())
    return tuple(sorted(palabras))


def detectar_loop(traza, ventana=6):
    if type(ventana) is not int or ventana < 1:
        raise ValueError("ventana debe ser un entero positivo")
    vistos = set()
    for evento in traza[-ventana:]:
        if evento["tipo"] != "action":
            continue
        clave = (evento["tool"], normalizar_input(evento["input"]))
        if clave in vistos:
            return True
        vistos.add(clave)
    return False


detector_resultados = {nombre: detectar_loop(traza) for nombre, traza in {
    "sana": TRAZA_SANA, "loop": TRAZA_LOOP, "fabrica": TRAZA_FABRICA
}.items()}
print(detector_resultados)


{'sana': False, 'loop': True, 'fabrica': False}


**Limitación honesta:** la normalización por palabras no atrapa paráfrasis
("cotización de NimbusSoft"). En producción se combina con: límite de pasos,
límite de llamadas a la MISMA tool, y similitud de embeddings entre acciones.

## Parte 3 — Presupuesto: el seguro que nunca falla

### ✏️ Ejercicio 3.1
Envuelve un agente con **presupuesto de pasos** y con detección de repetición: si agota el
presupuesto o repite la misma llamada, se detiene y lo dice en la traza.

No es una precaución teórica. En la sesión 11, con la H200 de verdad, el agente pidió
`estadisticas_ventas` con `2025-03`, luego con `2024-03`, y siguió probando años porque el
error de la herramienta no decía qué meses existían: **el presupuesto fue lo único que lo
detuvo**. Un agente sin tope no falla, se queda dando vueltas y gastando.

## 5. Presupuesto: decidir si una acción puede ejecutarse

La solución envuelve una lista de acciones simuladas con tres condiciones externas: máximo de pasos, costo disponible y repetición. La ejecución debe detenerse con un motivo explícito en su traza.

```mermaid
flowchart TD
    A[Proxima accion propuesta] --> P{Quedan pasos}
    P -->|no| F[Parar y registrar motivo]
    P -->|si| C{Alcanza presupuesto}
    C -->|no| F
    C -->|si| R{Repite una accion reciente}
    R -->|si| F
    R -->|no| E[Simular ejecucion y observar]
    E --> G[Actualizar pasos y gasto]
    G --> A
```

`pasos`, `gastado` y `traza` empiezan en cero, cero y lista vacía. La función interna `detener` usa esos valores actuales para registrar una parada y devolver el resumen. Ese acceso a variables de la función externa se llama cierre; permite no repetir el código de salida en cada condición.

### Las comprobaciones ocurren antes de ejecutar

`pasos >= max_pasos` impide dar un paso extra. `gastado + costo_paso > presupuesto` comprueba si alcanza para el siguiente paso; comprobar solo lo ya gastado permitiría exceder el presupuesto.

Se construye una propuesta con `ejecutada=False` y se compara junto con la traza existente. Si es repetida, se registra como bloqueada; **no se ejecuta ni se cobra**. Si pasa, se agrega como `ejecutada=True`, se suma el costo y se crea una observación simulada.

`{**accion, "ejecutada": True}` copia el diccionario de la acción y agrega o reemplaza una clave. No llama una función: aquí `**` está dentro de un diccionario.

### Tres salidas que debes poder calcular a mano

| Caso | Cálculo | Resultado |
| --- | --- | --- |
| Presupuesto 7, costo 2 por paso | Gastos 2, 4, 6; el cuarto requeriría 8. | Parada por presupuesto tras 3 pasos, gasto 6. |
| Presupuesto 100, máximo 5 | El costo no corta antes; se bloquea el sexto paso. | Parada por `max_pasos` tras 5 pasos, gasto 10. |
| Búsquedas repetidas normalizadas | Se realiza la primera; se bloquea la segunda propuesta. | Parada por repetición tras 1 paso, gasto 2. |

Los 12 pasos deseados se simulan con 12 consultas distintas. Las búsquedas repetidas son otro escenario: así puedes verificar cada condición por separado.

### Leer la construcción de las acciones y la parada

`range(1, 13)` produce los números del 1 al 12; el extremo 13 no se incluye. La comprensión de lista crea un diccionario por número y `f"consulta {i}"` inserta ese número en el texto. Por eso las 12 entradas son diferentes y el detector no corta este escenario.

`[e for e in TRAZA_LOOP if e["tipo"] == "action"]` conserva solo las acciones de la traza original. En este caso no conserva pensamientos ni observaciones; el controlador crea sus propias observaciones al simular cada ejecución.

| Propuesta en el caso de costo | Valores antes de proponerla | Comprobación | Qué cambia |
| --- | --- | --- | --- |
| Primera | `pasos=0`, `gastado=0`. | `0 + 2 <= 7`. | Ejecuta: pasos 1, gasto 2. |
| Segunda | `pasos=1`, `gastado=2`. | `2 + 2 <= 7`. | Ejecuta: pasos 2, gasto 4. |
| Tercera | `pasos=2`, `gastado=4`. | `4 + 2 <= 7`. | Ejecuta: pasos 3, gasto 6. |
| Cuarta | `pasos=3`, `gastado=6`. | `6 + 2 > 7`. | Registra parada; conserva pasos 3 y gasto 6. |

`traza + [propuesta]` crea una lista temporal para evaluar la acción antes de aceptarla. No modifica `traza`. `traza.append(...)` sí modifica la lista. `pasos += 1` equivale a `pasos = pasos + 1`.

Las comprobaciones tienen prioridad: primero pasos, luego costo y luego repetición. Si dos límites fallan simultáneamente, se registra el primero que encuentre el código. Si todas las acciones terminan justo en el límite, el estado es `completado`: ya no hay otra propuesta que bloquear.

Las salidas principales que debes ver son:

```text
costo -> presupuesto pasos: 3 gasto: 6
pasos -> max_pasos pasos: 5 gasto: 10
repeticion -> repeticion pasos: 1 gasto: 2
```

Los mensajes originales «seguro que nunca falla» y «un agente sin tope no falla» son énfasis didáctico. Estos límites contienen las acciones que pasan por este controlador; no resuelven una llamada de herramienta que nunca retorna. Para esa situación haría falta además un timeout.

Aquí un paso equivale a una acción simulada ejecutada. En la práctica del lunes un paso cuenta una decisión del simulador. Los nombres de los contadores son parecidos, pero debes definir qué cuentan antes de compararlos.

El costo usa **unidades ficticias**, no precios ni tokens de un servicio real. Para acotar consumo real habría que contar todas las llamadas del sistema, estimar o reservar su costo y registrar el consumo informado por el proveedor.


In [5]:
def controlar_agente(acciones, max_pasos=5, presupuesto=7, costo_paso=2):
    if type(max_pasos) is not int or max_pasos < 1:
        raise ValueError("max_pasos debe ser positivo")
    if type(presupuesto) is not int or presupuesto < 0:
        raise ValueError("presupuesto debe ser un entero no negativo")
    if type(costo_paso) is not int or costo_paso < 1:
        raise ValueError("costo_paso debe ser positivo")
    pasos, gastado, traza = 0, 0, []

    def detener(motivo):
        traza.append({"tipo": "parada", "motivo": motivo})
        return {"estado": motivo, "pasos": pasos, "gastado": gastado, "traza": traza}

    for accion in acciones:
        if pasos >= max_pasos:
            return detener("max_pasos")
        if gastado + costo_paso > presupuesto:
            return detener("presupuesto")
        propuesta = {**accion, "ejecutada": False}
        if detectar_loop(traza + [propuesta]):
            traza.append(propuesta)
            return detener("repeticion")
        # Este es el punto donde se ejecutaría una herramienta real.
        # Aquí solo simulamos una observación y un gasto fijo.
        pasos += 1
        gastado += costo_paso
        traza.append({**accion, "ejecutada": True, "paso": pasos})
        traza.append({"tipo": "observation", "texto": "Resultado simulado",
                      "gasto_acumulado": gastado})
    return {"estado": "completado", "pasos": pasos, "gastado": gastado, "traza": traza}


acciones_doce = [{"tipo": "action", "tool": "buscar", "input": f"consulta {i}"}
                for i in range(1, 13)]
acciones_repetidas = [e for e in TRAZA_LOOP if e["tipo"] == "action"]
por_costo = controlar_agente(acciones_doce)
por_pasos = controlar_agente(acciones_doce, presupuesto=100)
por_repeticion = controlar_agente(acciones_repetidas, presupuesto=100)
for nombre, caso in [("costo", por_costo), ("pasos", por_pasos), ("repeticion", por_repeticion)]:
    print(nombre, "->", caso["estado"], "pasos:", caso["pasos"], "gasto:", caso["gastado"])
    print("último evento:", caso["traza"][-1])


costo -> presupuesto pasos: 3 gasto: 6
último evento: {'tipo': 'parada', 'motivo': 'presupuesto'}
pasos -> max_pasos pasos: 5 gasto: 10
último evento: {'tipo': 'parada', 'motivo': 'max_pasos'}
repeticion -> repeticion pasos: 1 gasto: 2
último evento: {'tipo': 'parada', 'motivo': 'repeticion'}


## Parte 4 — Mini-Reflexion con verificador

La tarea: producir un JSON con formato exacto. El "verificador" es código
determinista (como los tests de un code agent). Si falla, la crítica entra al
contexto y el agente reintenta.

## 6. Verificador: separar JSON válido de un objeto que cumple el contrato

La salida exigida es un objeto con exactamente `categoria` y `urgencia`, categoría `tecnico` o `ventas`, y urgencia entera. El material no define un rango para urgencia; esta solución no añade uno.

El verificador devuelve una pareja `(ok, critica)`: un booleano y un mensaje. `ok, critica = verificador(respuesta)` separa esa pareja en dos variables.

| Comprobación | Qué evita |
| --- | --- |
| `json.loads(respuesta)` | Texto extra o sintaxis JSON incorrecta. |
| `isinstance(datos, dict)` | Aceptar arrays, `null` u otros tipos de raíz. |
| `set(datos) == {"categoria", "urgencia"}` | Campos faltantes o adicionales. |
| Categoría de tipo texto y dentro del conjunto | Categorías no admitidas o valores del tipo incorrecto. |
| `type(datos["urgencia"]) is int` | Decimales y booleanos. |

El original no comprueba si la raíz es un objeto y puede lanzar `AttributeError` para `[]` o `null`. También usa `isinstance(valor, int)`, que acepta `True` porque en Python `bool` es subclase de `int`. La copia corrige ambos casos y comprueba el tipo de categoría antes de buscarlo en un conjunto.

`"técnico"` es distinto de `"tecnico"` en una enumeración. Rechazarlo aquí significa incumplir el contrato, no escribir mal español.

Este verificador comprueba formato y dominio de campos. No recibe un ticket ni una respuesta de referencia, así que no prueba que la categoría o urgencia sean las correctas para una situación real.

### Qué sucede con una entrada concreta

Para `'{"categoria": "tecnico", "urgencia": 4}'`, `json.loads` crea el diccionario Python `{"categoria": "tecnico", "urgencia": 4}`. `set(datos)` toma sus **claves**, no sus valores. El orden de las claves no importa: se acepta también un objeto que escriba `urgencia` antes de `categoria`.

Un candidato con `urgencia: 4.0` es JSON válido, pero se convierte en un `float`, así que incumple el requisito de entero. Un candidato con `urgencia: true` se convierte en `True`, de tipo `bool`, y también se rechaza. El `try/except` permite devolver una crítica cuando el texto no se puede interpretar como JSON, en vez de detener toda la práctica con una excepción.

La condición de categoría usa `or`: si el valor no es texto, ya se rechaza y Python no evalúa la segunda parte. Así una lista como categoría no llega a la operación de pertenencia en un conjunto, que necesita valores utilizables como claves. Cada `return False, ...` describe el primer fallo encontrado; no enumera todos los errores del candidato a la vez.


In [6]:
def verificador(respuesta):
    try:
        datos = json.loads(respuesta)
    except (json.JSONDecodeError, TypeError):
        return False, "Entrega solo JSON válido, sin texto adicional"
    if not isinstance(datos, dict):
        return False, "La raíz debe ser un objeto JSON"
    if set(datos) != {"categoria", "urgencia"}:
        return False, "Usa exactamente las claves categoria y urgencia"
    if not isinstance(datos["categoria"], str) or datos["categoria"] not in {"tecnico", "ventas"}:
        return False, "categoria debe ser tecnico o ventas, sin tilde"
    if type(datos["urgencia"]) is not int:
        return False, "urgencia debe ser un entero, no booleano ni decimal"
    return True, "OK"


INTENTOS_SIMULADOS = [
    'La categoría es técnica con urgencia 4: {"categoria": "tecnico", "urgencia": 4}',  # texto extra
    '{"categoria": "técnico", "urgencia": 4}',                                          # tilde: fuera de dominio
    '{"categoria": "tecnico", "urgencia": 4}',                                          # ✓
]


## Parte 4 — Reflexion: reintentar leyendo la crítica

### ✏️ Ejercicio 4.1
Implementa `reflexion(max_intentos=3)`: genera una salida, **verifícala**, y si falla,
vuelve a generar pasándole la crítica de lo que falló. Devuelve la primera salida que pase
y la traza de los intentos.

Lo que hay que mirar al correrlo no es que acabe pasando: es **de qué depende que pase**.
Reflexion solo funciona si el verificador dice algo accionable. «Está mal» no mejora nada;
«falta el campo `total`» sí. Es la misma lección que el martes de la semana 2 con los
mensajes de error de una herramienta, y la misma que dejó la sesión 11 al ver al modelo
adivinar años porque el error no decía qué meses existían.

## 7. Mini-Reflexion: la crítica debe entrar en la siguiente generación

`INTENTOS_SIMULADOS` contiene tres salidas candidatas: una con texto extra, otra con la categoría fuera del dominio y otra válida. Recorrer esa lista automáticamente no demostraría que el generador leyó el feedback.

En la solución, `generar_con_criticas` recibe las críticas anteriores y usa la última para elegir una salida. No recibe el número de intento. Así queda visible qué información provocó el cambio.

```mermaid
flowchart LR
    G[Generar con criticas previas] --> V[Verificar candidato]
    V -->|aceptado| F[Devolver salida]
    V -->|rechazado| C[Guardar critica accionable]
    C -->|quedan intentos| G
    C -->|agotados| P[Parar sin salida aceptada]
```

### Seguir las variables del bucle

`criticas=[]` conserva el feedback acumulado. `traza=[]` conserva qué salió en cada intento y qué críticas recibió el generador. `list(criticas)` pasa una copia de la lista para que el registro no cambie cuando se añadan críticas después.

En cada vuelta, se genera un candidato, se llama al verificador y se registra el resultado. Si `ok` es verdadero, `return` termina inmediatamente: no se sigue corrigiendo una respuesta ya aceptada. Si es falso, `criticas.append(critica)` añade el mensaje que verá la próxima generación.

| Intento | Críticas recibidas | Salida y resultado |
| --- | --- | --- |
| 1 | Ninguna. | Texto antes del JSON; se pide entregar solo JSON. |
| 2 | Crítica sobre JSON. | `"técnico"`; se pide usar categoría admitida sin tilde. |
| 3 | Las dos críticas. | `{"categoria": "tecnico", "urgencia": 4}`; aceptado. |

Con `max_intentos=2`, la salida es `intentos_agotados`, con `respuesta=None`: dos intentos no alcanzan en este guion. Un generador que ignora siempre las críticas también agota el límite. Si el primer candidato ya es válido, termina en un intento.

El generador es una regla local escrita para enseñar el mecanismo. No se midió si un modelo real mejora, ni se entrenaron sus pesos. La «memoria» aquí es la lista de críticas que se conserva dentro de la corrida.

### Qué se conserva entre intentos

En el primer intento, `criticas=[]`; el generador devuelve texto extra y el verificador lo rechaza. Después de `criticas.append(critica)`, la lista contiene una instrucción de formato. En el segundo intento se **entrega esa lista al generador**; su condición `"solo JSON" in criticas[-1]` reconoce el feedback y elige otra salida. `[-1]` significa último elemento. La segunda crítica permite elegir la tercera salida.

`generador` es una función que se puede pasar como argumento. La comprobación `reflexion(generador=lambda criticas: INTENTOS_SIMULADOS[0])` reemplaza el generador por uno que devuelve siempre la primera salida. `lambda` define aquí una función breve: recibe `criticas`, pero las ignora. El resultado es `intentos_agotados`, aunque el verificador siga dando consejos. Ese contraste comprueba que el éxito de esta simulación depende de usar el feedback.

`list(criticas)` crea una copia en dos lugares: al llamar al generador y al guardar cada evento. Si se guardara la misma lista compartida, las críticas añadidas después aparecerían también en registros anteriores y parecería que el primer intento recibió información del futuro. La primera fila de la traza debe conservar `criticas_recibidas: []`.

`None` representa ausencia de respuesta aceptada cuando se agotan los intentos. La traza conserva candidatos rechazados para inspeccionarlos, pero el campo `respuesta` no los presenta como una solución válida.


In [7]:
def generar_con_criticas(criticas):
    # La crítica recibida participa en la elección: no usamos el número de intento.
    if not criticas:
        return INTENTOS_SIMULADOS[0]
    if "solo JSON" in criticas[-1]:
        return INTENTOS_SIMULADOS[1]
    if "sin tilde" in criticas[-1]:
        return INTENTOS_SIMULADOS[2]
    return INTENTOS_SIMULADOS[0]


def reflexion(max_intentos=3, generador=None):
    if type(max_intentos) is not int or max_intentos < 1:
        raise ValueError("max_intentos debe ser positivo")
    if generador is None:
        generador = generar_con_criticas
    criticas, traza = [], []
    for intento in range(1, max_intentos + 1):
        respuesta = generador(list(criticas))
        ok, critica = verificador(respuesta)
        traza.append({"intento": intento, "respuesta": respuesta,
                      "criticas_recibidas": list(criticas), "ok": ok, "critica": critica})
        if ok:
            return {"estado": "verificado", "respuesta": respuesta, "traza": traza}
        criticas.append(critica)
    return {"estado": "intentos_agotados", "respuesta": None, "traza": traza}


reflexion_resultado = reflexion()
print(json.dumps(reflexion_resultado, ensure_ascii=False, indent=2))


{
  "estado": "verificado",
  "respuesta": "{\"categoria\": \"tecnico\", \"urgencia\": 4}",
  "traza": [
    {
      "intento": 1,
      "respuesta": "La categoría es técnica con urgencia 4: {\"categoria\": \"tecnico\", \"urgencia\": 4}",
      "criticas_recibidas": [],
      "ok": false,
      "critica": "Entrega solo JSON válido, sin texto adicional"
    },
    {
      "intento": 2,
      "respuesta": "{\"categoria\": \"técnico\", \"urgencia\": 4}",
      "criticas_recibidas": [
        "Entrega solo JSON válido, sin texto adicional"
      ],
      "ok": false,
      "critica": "categoria debe ser tecnico o ventas, sin tilde"
    },
    {
      "intento": 3,
      "respuesta": "{\"categoria\": \"tecnico\", \"urgencia\": 4}",
      "criticas_recibidas": [
        "Entrega solo JSON válido, sin texto adicional",
        "categoria debe ser tecnico o ventas, sin tilde"
      ],
      "ok": true,
      "critica": "OK"
    }
  ]
}


El patrón completo: **generar → verificar → criticar → regenerar con la crítica**.
Funciona porque el verificador aporta información que el modelo no tenía.
Sin verificador confiable, Reflexion degenera en el modelo dándose la razón.

## 8. Reflexión para un proyecto: una respuesta de ejemplo

Para un agente que consulta información, ReAct simple puede servir como baseline porque la siguiente consulta depende de lo observado. Primero conviene medir qué resuelve ese bucle con límites y trazas, antes de añadir coordinación compleja.

Para reintentos, una señal verificable podría ser el parseo JSON, el conjunto de campos y sus tipos. Si la tarea exige clasificar tickets correctamente, además necesitas casos etiquetados o reglas de dominio. Un objeto bien formado puede contener una clasificación equivocada.

Esta respuesta es un ejemplo de razonamiento sobre arquitectura; no presupone el contenido completo de tu Lab 03.


In [8]:
# Ejemplo de decisión de proyecto, no requisitos asumidos de Lab 03:
# a) Empezaría con ReAct simple para decidir la consulta según los datos observados,
#    con límite externo de pasos y registro de cada solicitud y resultado.
# b) Para Reflexion usaría señales verificables: JSON parseable, campos exactos,
#    tipos y valores permitidos. La clasificación correcta exige además ejemplos
#    etiquetados o criterios de dominio; pasar formato no prueba verdad.


## Parte 5 — Leer una lista de patrones con criterio

Circula mucho contenido del tipo "los N patrones de agentes que todo ingeniero
debe conocer". Suelen ser útiles **y** estar mal ordenados: mezclan cosas que
viven en niveles distintos. Saber separarlas es la habilidad de hoy.

Una lista publicada recientemente enumeraba estos diez:

`ReAct` · `Chain of Thought` · `Parallel Execution` · `Self-Critique` ·
`Agent Handoff` · `Function Calling` · `LLM Workflow` · `Task Decomposition` ·
`Episodic Memory` · `Self-Reflection`

## 9. Clasificar los diez términos sin mezclar niveles

La consigna exige una cubeta principal por elemento. Esta es una solución posible, indicando cuándo depende de cómo se implemente:

| Término | Cubeta principal para esta actividad | Motivo |
| --- | --- | --- |
| ReAct | Control. | Organiza decisión, acción y observación. |
| Parallel Execution | Control. | Organiza tareas independientes simultáneamente. |
| Agent Handoff | Control. | Transfiere responsabilidad y contexto. |
| LLM Workflow | Control. | Nombre amplio para una secuencia organizada. |
| Task Decomposition | Control, si coordina subproblemas. | Divide y organiza el trabajo; puede ser solo prompting en otra implementación. |
| Chain of Thought | Prompting. | Cambia cómo se pide elaborar una respuesta. |
| Self-Critique | Prompting, como instrucción de revisión. | Pide evaluar un candidato; puede integrarse en un controlador. |
| Self-Reflection | Prompting, como instrucción de reflexión. | Pide extraer una lección; puede incorporar memoria y control. |
| Function Calling | Mecanismo. | Estructura una petición de función. |
| Episodic Memory | Memoria. | Conserva experiencias de corridas. |

La pareja que probablemente busca la consigna es Self-Critique y Self-Reflection. Algunos autores las usan como sinónimos, pero no son universalmente idénticas: revisar una salida y guardar una lección para otra ejecución pueden ser operaciones distintas. La lista fuente no está adjunta; no se puede comprobar que sus implementaciones sean exactamente iguales.

Que el autor venda memoria puede explicar su énfasis. Para evaluar la lista hay que contrastar definiciones, ejemplos de código, señales de aceptación y mediciones, en vez de aceptar o rechazar todo por su afiliación.


In [9]:
# Una cubeta principal por término, siguiendo la simplificación del ejercicio:
# 1) Control: ReAct, Parallel Execution, Agent Handoff, LLM Workflow,
#    Task Decomposition (si además coordina la ejecución de los subproblemas).
# 2) Prompting: Chain of Thought, Self-Critique, Self-Reflection
#    (entendidos aquí como instrucciones para revisar una salida).
# 3) Mecanismo: Function Calling.
# 4) Memoria: Episodic Memory.
# b) La pareja que probablemente pretende la consigna es Self-Critique y
#    Self-Reflection. Se usan como sinónimos en algunas listas; no son idénticos
#    universalmente y no bastan para implementar el patrón Reflexion del paper.
# c) Vender memoria puede explicar el énfasis, pero no valida ni refuta la lista.
#    Contrastaría definiciones, código ejecutable, criterios de evaluación y costos.


## Cierre

| Patrón | Compra | Cuesta | Protección asociada |
|---|---|---|---|
| ReAct | exploración anclada en hechos | 1 llamada/paso | detector de loops + presupuesto |
| Plan-Execute | coherencia y revisión del plan | rigidez | replanning |
| Reflexion | mejora con feedback | ×2–4 llamadas | verificador REAL |
| Task Decomposition | subproblemas manejables | coordinación y estado | límite de profundidad |
| Parallel Execution | latencia | costo simultáneo, orden no garantizado | presupuesto global |
| Agent Handoff | especialización | pérdida de contexto en el traspaso | contrato de traspaso explícito |

Chain of Thought, function calling y memoria episódica **no** entran en esta
tabla: no son patrones de control. Ver Parte 5.

**Mañana:** cuándo graduarte del loop artesanal a un grafo orquestado
(LangGraph) y qué es MCP.

## Comprobaciones de los resultados

Estas condiciones deben cumplirse al ejecutar las celdas en orden. `assert` detiene la ejecución si encuentra una condición falsa. Los casos deliberadamente erróneos se comprueban como errores, no como respuestas válidas.


In [10]:
assert detector_resultados == {"sana": False, "loop": True, "fabrica": False}
casos_ventana = {
    "loop_ultimos_3_eventos": detectar_loop(TRAZA_LOOP, ventana=3),
    "loop_ultimos_6_eventos": detectar_loop(TRAZA_LOOP, ventana=6),
    "misma_entrada_otra_tool": detectar_loop([
        {"tipo": "action", "tool": "buscar", "input": "precio acción NimbusSoft"},
        {"tipo": "action", "tool": "cotizar", "input": "NimbusSoft precio acción"},
    ]),
    "traza_vacia": detectar_loop([]),
}
assert casos_ventana == {"loop_ultimos_3_eventos": False,
                        "loop_ultimos_6_eventos": True,
                        "misma_entrada_otra_tool": False, "traza_vacia": False}
assert normalizar_input("precio acción NimbusSoft") == normalizar_input("NimbusSoft precio acción")
assert normalizar_input("Ana paga a Luis") == normalizar_input("Luis paga a Ana")  # Limitación.
assert por_costo["estado"] == "presupuesto" and por_costo["gastado"] == 6 and por_costo["pasos"] == 3
assert por_pasos["estado"] == "max_pasos" and por_pasos["pasos"] == 5
assert por_repeticion["estado"] == "repeticion" and por_repeticion["pasos"] == 1
assert por_repeticion["traza"][-2]["ejecutada"] is False
sin_presupuesto = controlar_agente(acciones_doce, presupuesto=0)
assert sin_presupuesto["pasos"] == sin_presupuesto["gastado"] == 0
completo = controlar_agente(acciones_doce[:2], max_pasos=2, presupuesto=4)
assert completo["estado"] == "completado" and completo["gastado"] == 4
casos_verificador = {respuesta: verificador(respuesta) for respuesta in [
    "[]", "null", "no es JSON", '{"categoria": [], "urgencia": 4}',
    '{"categoria": "tecnico", "urgencia": true}',
    '{"categoria": "tecnico", "urgencia": 4.0}',
    '{"categoria": "técnico", "urgencia": 4}',
    '{"categoria": "tecnico", "urgencia": 4, "extra": 1}',
    '{"categoria": "tecnico", "urgencia": 4}',
]}
assert sum(ok for ok, _ in casos_verificador.values()) == 1
assert reflexion_resultado["estado"] == "verificado" and len(reflexion_resultado["traza"]) == 3
assert [evento["ok"] for evento in reflexion_resultado["traza"]] == [False, False, True]
assert reflexion_resultado["traza"][0]["criticas_recibidas"] == []
assert reflexion_resultado["traza"][1]["criticas_recibidas"] == [reflexion_resultado["traza"][0]["critica"]]
assert reflexion_resultado["traza"][2]["criticas_recibidas"] == [
    reflexion_resultado["traza"][0]["critica"], reflexion_resultado["traza"][1]["critica"]]
agotado = reflexion(max_intentos=2)
assert agotado["estado"] == "intentos_agotados" and agotado["respuesta"] is None
sin_mejora = reflexion(generador=lambda criticas: INTENTOS_SIMULADOS[0])
assert sin_mejora["estado"] == "intentos_agotados"
temprano = reflexion(generador=lambda criticas: INTENTOS_SIMULADOS[2])
assert temprano["estado"] == "verificado" and len(temprano["traza"]) == 1
resultados_verificados = {
    "tipo": "simulacion local; no ejecuta busquedas ni evalua un LLM",
    "detector": detector_resultados, "ventana_y_herramienta": casos_ventana,
    "corte_por_costo": por_costo, "corte_por_pasos": por_pasos,
    "corte_por_repeticion": por_repeticion,
    "verificador": casos_verificador,
    "reflexion": reflexion_resultado,
    "reflexion_agotada": agotado,
    "generador_ignora_feedback": sin_mejora,
    "aceptacion_temprana": temprano, "sin_presupuesto": sin_presupuesto,
    "completado_en_limite": completo,
    "verificacion": "Todas las comprobaciones del martes pasaron",
}
print(resultados_verificados["verificacion"])


Todas las comprobaciones del martes pasaron
